# 17 — Modal Analysis of a Cantilever Beam

**Curriculum slot:** Tier 5, slot 17.  
**Prerequisite:** 02 — 2D Cantilever Beam, 06 — Sections Catalog.  
**Strategy used for results:** `capture_live_modes(...)` — the modes of a
plain openseespy deck, solved on the live domain and written as native
mode stages (no recorder route records modes).

## Purpose

Tier 5 covers the four major analysis types beyond static-linear:
**modal**, **buckling**, **pushover**, **time-history**. This is the
modal entry point.

A linear-elastic cantilever of length $L$ with mass per unit length
$\bar{m} = \rho A$ has natural circular frequencies

$$
\omega_n \;=\; (\beta_n L)^{2} \sqrt{\dfrac{E I}{\bar{m}\,L^{4}}},
$$

where $\beta_n L$ satisfies $\cos(\beta L)\cosh(\beta L) + 1 = 0$.
First three roots:

| Mode | $\beta_n L$ | $\omega_n / \omega_1$ |
|---|---|---|
| 1 | 1.8751 | 1.000 |
| 2 | 4.6941 | 6.267 |
| 3 | 7.8548 | 17.55 |

## Why this notebook is interesting

* **Modes come from the live domain.** No recorder route records
  modes: `spec.emit_recorders` and `spec.emit_default` both refuse a
  modal record. This notebook builds its deck with plain `ops.*`
  calls, so it calls `capture_live_modes(...)`, which runs
  `ops.eigen(N)` on the live domain and writes one stage per mode.
  (A model built through the bridge would use `apeSees.eigen(N)`
  and its `EigenResult.results()` instead.)
* **Read back via `results.modes`.** Each mode is a stage with
  `kind="mode"`, exposing `mode_index`, `eigenvalue`, `frequency_hz`,
  `period_s` as scoped properties.
* Mode shapes are slabs with `T = 1` (one "step" per mode) — same
  read API, just `mode.nodes.get(component="displacement_z")`.

## 1. Imports and parameters

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import openseespy.opensees as ops

from apeGmsh import apeGmsh, Results
from apeGmsh.opensees import OpenSeesModel
from apeGmsh.results.live import capture_live_modes

# Geometry / section
L  = 3.0
E  = 2.1e11
nu = 0.3
G  = E / (2.0 * (1.0 + nu))
A, Iy, Iz, J = 1.0e-3, 1.0e-5, 1.0e-5, 2.0e-5
rho   = 7850.0          # steel
m_bar = rho * A          # mass per unit length

# Mesh density — higher than the static cantilever so the higher modes
# get enough segments per half-wavelength.
N_ELEM  = 30
LC      = L / N_ELEM
N_MODES = 3

## 2. Geometry + mesh

A line cantilever — same as slot 02 but with a transfinite curve
so the element count is exactly `N_ELEM`.

In [2]:
from apeGmsh import workdir
g_ctx = apeGmsh(model_name="17_modal", save_to=workdir() / "17_modal.h5", verbose=False)
g = g_ctx.__enter__()

p_base = g.model.geometry.add_point(0.0, 0.0, 0.0, lc=LC)
p_tip  = g.model.geometry.add_point(L,   0.0, 0.0, lc=LC)
ln     = g.model.geometry.add_line(p_base, p_tip)
g.model.sync()

g.physical.add(0, [p_base], name="base")
g.physical.add(0, [p_tip],  name="tip")
g.physical.add(1, [ln],     name="beam")

g.mesh.structured.set_transfinite_curve(ln, N_ELEM + 1)
g.mesh.generation.generate(1)
fem = g.mesh.queries.get_fem_data()
print(f"mesh: {fem.info.n_nodes} nodes, {fem.info.n_elems} elements")

C:\Users\nmb\AppData\Local\Temp\ipykernel_55852\2080555.py:2: UnitNotDeclaredWarning: apeGmsh(...): no consistent unit system declared (pass units='kN_m_s' or a baseUnits.systems module such as units=kN_m_s); Results and apeGmshViewer will require manual unit selection for unit conversions.
  g_ctx = apeGmsh(model_name="17_modal", save_to=workdir() / "17_modal.h5", verbose=False)


mesh: 31 nodes, 32 elements


## 3. Build the OpenSees model — vanilla openseespy

Standard cantilever, plus restraint of every non-bending DOF on every
non-base node. Without that, axial / out-of-plane modes interleave
with the bending modes and break the ordering vs the analytical
Euler-Bernoulli reference.

In [3]:
ops.wipe()
ops.model("basic", "-ndm", 3, "-ndf", 6)

for nid, xyz in fem.nodes.select():
    ops.node(int(nid), float(xyz[0]), float(xyz[1]), float(xyz[2]))

ops.geomTransf("Linear", 1, 0.0, 1.0, 0.0)
for group in fem.elements.select(target="beam").result():
    for eid, conn in zip(group.ids, group.connectivity):
        ops.element(
            "elasticBeamColumn", int(eid),
            int(conn[0]), int(conn[1]),
            A, E, G, J, Iy, Iz, 1,
        )

# fix the base fully
for n in fem.nodes.select(target="base").ids:
    ops.fix(int(n), 1, 1, 1, 1, 1, 1)

# Free DOFs on non-base nodes: u_z (3), r_y (5) — pure in-plane bending
base_set = {int(n) for n in fem.nodes.select(target="base").ids}
for nid, _ in fem.nodes.select():
    if int(nid) in base_set:
        continue
    ops.fix(int(nid), 1, 1, 0, 1, 0, 1)

## 4. Lumped-mass assembly

Each non-base node gets the consistent-tributary lumped mass
$m_{\text{full}} = \bar{m}\,h$; the tip is a half-tribute. Rotational
inertia is negligible for a slender beam.

In [4]:
h = L / N_ELEM
m_full = m_bar * h
m_half = m_full / 2.0

tip_node = int(next(iter(fem.nodes.select(target="tip").ids)))
for nid in fem.nodes.ids:
    nid = int(nid)
    if nid in base_set:
        continue
    m_here = m_half if nid == tip_node else m_full
    ops.mass(nid, m_here, m_here, m_here, 0.0, 0.0, 0.0)

## 5. Capture the modes from the live domain

`capture_live_modes(model_h5, N_MODES, path, ops=ops)` solves
`ops.eigen(N_MODES)` on the deck in `ops` and writes one native stage per
mode (`kind="mode"`, with its eigenvalue, frequency and period). It pairs
every FEM node with the live node of its tag, so it needs the FEMData the
deck was built from: `fem.to_h5(...)` writes it as a model archive.

`ops.system("BandGeneral")` comes first. On an indefinite stiffness (a
negative spring, a softened member) the stock default eigensolver can
return plausible positive eigenvalues without raising, and a
`BandGeneral` system gives the true ones. This cantilever's stiffness is
positive definite, but the line costs nothing and keeps the cell safe to
copy. A deck that solves eigen a second time must
call `ops.wipeAnalysis()`, then `ops.system("BandGeneral")` again, before `capture_live_modes`:
`wipeAnalysis()` also drops the system, and the default solver comes back
with it.

In [5]:
from apeGmsh import workdir
OUT = workdir()
model_h5 = OUT / "model.h5"
fem.to_h5(str(model_h5))              # the FEMData the deck was built from

In [6]:
ops.system("BandGeneral")             # safe on an indefinite stiffness
results_path = capture_live_modes(model_h5, N_MODES, OUT / "modes.h5", ops=ops)
print(f"wrote {results_path} ({results_path.stat().st_size / 1024:.1f} KB)")

wrote outputs\modes.h5 (64.4 KB)


## 6. Read modes via `results.modes`

Each entry is a `Results` scoped to one mode stage with `mode_index`,
`eigenvalue`, `frequency_hz`, `period_s` as scoped properties — no
need to know the underlying HDF5 layout.

In [7]:
results = Results.from_native(
    results_path,
    fem=fem,
    model=OpenSeesModel.from_h5(model_h5),
)
print(results.inspect.summary())

Results: outputs\modes.h5
  FEM: 31 nodes, 32 elements (snapshot_id=2bb96fd202593c3ec28065454e4d6557)
  Stages (3):
    - stage_0 (mode_1): steps=1, kind=mode, f=32.14 Hz, T=0.03111 s, mode_index=1
    - stage_1 (mode_2): steps=1, kind=mode, f=201.2 Hz, T=0.004971 s, mode_index=2
    - stage_2 (mode_3): steps=1, kind=mode, f=562.7 Hz, T=0.001777 s, mode_index=3

In [8]:
modes = sorted(results.modes, key=lambda m: m.mode_index or 0)
print(f"{'mode':>4s}  {'ω [rad/s]':>14s}  {'f [Hz]':>14s}  {'T [s]':>14s}")
for m in modes:
    print(f"{m.mode_index:>4d}  {np.sqrt(m.eigenvalue):>14.6e}  "
          f"{m.frequency_hz:>14.6e}  {m.period_s:>14.6e}")

mode       ω [rad/s]          f [Hz]           T [s]
   1    2.019583e+02    3.214266e+01    3.111130e-02
   2    1.264059e+03    2.011812e+02    4.970644e-03
   3    3.535386e+03    5.626741e+02    1.777228e-03


## 7. Verification against analytical $\omega_n$

In [9]:
beta_L = np.array([1.87510407, 4.69409113, 7.85475744])
omega_analytical = (beta_L**2) * np.sqrt(E * Iz / (m_bar * L**4))

print(f"{'mode':>4s}  {'FEM ω':>14s}  {'analyt ω':>14s}  {'err %':>10s}")
for i, m in enumerate(modes):
    w_fem = np.sqrt(m.eigenvalue)
    w_an  = omega_analytical[i]
    err   = abs(w_fem - w_an) / w_an * 100.0
    print(f"{m.mode_index:>4d}  {w_fem:>14.4e}  {w_an:>14.4e}  {err:>9.4f} %")

mode           FEM ω        analyt ω       err %
   1      2.0196e+02      2.0206e+02     0.0510 %
   2      1.2641e+03      1.2663e+03     0.1767 %
   3      3.5354e+03      3.5457e+03     0.2900 %


## 8. Plot the mode shapes

Each mode is a stage with `T = 1` — pulling `displacement_z` returns
a single-row slab whose shape over $x$ is the mode shape. OpenSees
mass-normalises it ($\phi^{T} M \phi = 1$), so its largest entry is not
1: here $\max|u_z| \approx 0.41$. The plot flips each shape so the tip
is positive.

In [10]:
tag_to_idx = {int(t): i for i, t in enumerate(fem.nodes.ids)}

fig, ax = plt.subplots(figsize=(7, 5))
for i, m in enumerate(modes):
    slab = m.nodes.get(component="displacement_z")
    x = np.array([fem.nodes.coords[tag_to_idx[int(n)], 0]
                  for n in slab.node_ids])
    u = slab.values[0]
    order = np.argsort(x)
    sign = np.sign(u[order][-1]) or 1.0   # tip-positive convention
    ax.plot(
        x[order], sign * u[order],
        "o-", ms=3, color=f"C{i}",
        label=f"mode {m.mode_index}  ({m.frequency_hz:.1f} Hz)",
    )

ax.axhline(0.0, color="gray", lw=0.5)
ax.set_xlabel("x  [m]"); ax.set_ylabel("mass-normalised u_z  (φᵀMφ = 1)")
ax.set_title("Cantilever — first three bending mode shapes")
ax.legend(); ax.grid(True, alpha=0.3)
fig.tight_layout()

## 9. (Optional) Open the post-solve viewer

The viewer's stage selector picks per mode — switch through them to
see each shape rendered in the deformed-beam view.

In [11]:
results.viewer(blocking=False)


[skip viewer] APEGMSH_SKIP_VIEWER set


## What this unlocks

* **Modes from a hand-built deck.** One call to
  `capture_live_modes(model_h5, N, path)` writes N stages of
  `kind="mode"`; the read side surfaces them via `results.modes`.
* **Why no recorder route.** The classic recorder pipeline can't drive
  `ops.eigen(...)`, so `spec.emit_recorders` and `spec.emit_default`
  refuse a modal record. On a bridge-built model, `apeSees.eigen(N)`
  returns an `EigenResult` whose `.results()` does the same job.
* **Stage-scoped properties** — `mode_index`, `eigenvalue`,
  `frequency_hz`, `period_s` on each scoped Results.
* **Lumped-mass tributary pattern** — half-mass at the tip,
  full mass at internal nodes. Generalises to inhomogeneous beams
  and shells.

Next: notebook 19 — pushover with a fiber section. The first
nonlinear example of the curriculum.

In [12]:
g_ctx.__exit__(None, None, None)